# TRKR GUI Simulation Notebook

This notebook controls an **already open TRKR GUI** through the common
`kohdalab_control` SDK. It never starts a controller, opens an instrument or
serial port, or changes the GUI's default safety policy: every request is
executed by the GUI, which shows **External control** in its top banner with
**Stop** and **Take control** available throughout.

It runs only when **Demo is already selected in the GUI**. The notebook only
reads the source; it never switches between Demo and hardware. On any other
source it returns control and stops before starting anything.

* **Explicit endpoint** – set `ENDPOINT` (or `TRKR_CONTROL_ENDPOINT`) to the
  GUI's descriptor file. The notebook lists candidates but never picks one.
* **One protected scope** – acquisition, the Demo check, the plan, start and
  wait all run inside one `try`/`finally`, so control is always returned.
* **Explicit start ID** – `start` uses the caller UUID `START_REQUEST_ID`,
  which the GUI uses as the job ID. The notebook waits for *that* job only.
* **Lost acknowledgement** – if the start reply is lost, the notebook reads
  the receipt of `START_REQUEST_ID` (read-only). It never resends start and
  never sends an untargeted Stop.
* **Timeouts** – each request, the job wait, Stop and release have bounds. A
  timeout means the outcome is unknown; nothing is retried automatically.
* **Interrupt cleanup** – Kernel ▸ Interrupt (or a wait timeout) stops exactly
  this job and confirms its cleanup *before* releasing control. The original
  error is kept; cleanup errors are reported alongside it.
* **Stop on release** – for its own lease only, the notebook opts in to
  `stop_on_disconnect=True` and `stop_on_error=True` (the GUI defaults stay
  `false`). If a start whose reply was lost begins running just before
  release, or the kernel dies and the lease expires, the GUI stops that job
  instead of letting it continue.

The other notebooks in this folder are the legacy **headless** workflow: they
start their own Rust controller and do not use the open GUI. Do not run them
against instruments that a GUI is using.

開いているTRKR GUIを共通SDKで操作します。GUIで事前にDemoを選択してください（Notebookはソースを読み取るだけで切り替えません）。
接続先は`ENDPOINT`で明示し、開始要求UUIDで自分のジョブだけを待機・停止します。応答が失われた場合は受領記録で照合し、再送や対象なしの停止は行いません。
このリースに限り`stop_on_disconnect=True`と`stop_on_error=True`を指定し、解放時やリース失効時に実行中のジョブをGUIが停止します（GUIの既定値は変更しません）。
他のNotebookは独自のコントローラーを起動する従来のヘッドレス方式です。

In [ ]:
import json
import os
import sys
import tempfile
import time
import uuid
from pathlib import Path

def import_control_sdk():
    """Prefer the SDK shipped beside this notebook (source, portable, macOS)."""
    cwd = Path.cwd().resolve()
    for root in dict.fromkeys((cwd, cwd.parent, cwd.parent.parent, cwd.parent.parent.parent)):
        for directory in (root / "bindings" / "control", root / "common-sdk" / "control"):
            if (directory / "kohdalab_control.py").is_file():
                if str(directory) not in sys.path:
                    sys.path.insert(0, str(directory))
                break
    try:
        import kohdalab_control
    except ImportError as error:
        raise ImportError("Install the common SDK: python -m pip install ./bindings/control") from error
    return kohdalab_control

control = import_control_sdk()
ControlClient, ControlError = control.ControlClient, control.ControlError

# Safety gate kept for parity with the other notebooks: this notebook has no
# hardware path. Nothing starts unless the GUI already reports source "demo".
HARDWARE = False

# Explicit endpoint: the GUI's private descriptor, for example
# ~/.kohdalab/trkr/run/trkr-gui-<pid>-<instance>.json
ENDPOINT = os.environ.get("TRKR_CONTROL_ENDPOINT") or None

# Bounds (seconds). None are retried; a timeout leaves the outcome unknown.
REQUEST_TIMEOUT_S = 3.0    # one request/response
JOB_TIMEOUT_S = 60.0       # waiting for this job's terminal receipt
STOP_TIMEOUT_S = 15.0      # Stop until this job's cleanup is confirmed
RELEASE_TIMEOUT_S = 10.0   # returning control to the GUI
RECONCILE_TIMEOUT_S = 5.0  # reading the start receipt after a lost reply
POLL_INTERVAL_S = 0.1

WORKFLOW = "signal_monitor"
# A new file in the temporary directory; the operator's output settings and
# existing data are never used or overwritten.
OUTPUT_CSV = Path(tempfile.gettempdir()) / f"trkr-gui-simulation-{uuid.uuid4().hex[:8]}.csv"

if ENDPOINT is None:
    candidates = control.discover("kohdalab-trkr")
    listing = "\n".join(f"  {e.path}  (PID {e.pid}, instance {e.instance_id})" for e in candidates)
    raise RuntimeError(
        "Set ENDPOINT to the descriptor of the TRKR GUI to control.\n"
        + (f"Responsive TRKR GUIs:\n{listing}" if candidates else "No responsive TRKR GUI was found; open the GUI first.")
    )
print(f"Endpoint: {ENDPOINT}")

## Connect (read-only)

`describe()` and `status()` are read-only and do not lock the GUI.

In [ ]:
client = ControlClient("kohdalab-trkr", endpoint=ENDPOINT, timeout=REQUEST_TIMEOUT_S)
description = client.describe()
print(f"GUI {description['application_version']} · release policy: {description['safety'].get('on_release')}")
state = client.status()
print(f"Status: {json.dumps({k: state[k] for k in ('phase', 'busy', 'cleanup')})}")

## Acquire, check Demo, start, wait, clean up

Everything from `acquire()` on runs in one protected scope. `acquire()` locks
the GUI controls and shows the External control banner. The Demo check is a
read-only `config()`; select Demo in the GUI beforehand.

Interrupt the kernel to cancel. `finally` stops exactly `START_REQUEST_ID`
(only if it exists and is not terminal), waits for its own terminal receipt
with verified cleanup, and then releases control. The original error is
re-raised unchanged; any cleanup error is printed and attached to it as a
note. The GUI's Stop / Take control remain available throughout.

`acquire()` opts in to `stop_on_disconnect=True` and `stop_on_error=True`
for this lease only; the GUI's defaults (both `false`, `on_release:
continue`) are unchanged. This covers the one case the notebook cannot
target: if the start reply is lost and no receipt is found within
`RECONCILE_TIMEOUT_S`, the outcome is still **unknown** and the request may
still be applied. Should that job be running when control is released (or
when the lease expires because the kernel died), the GUI requests Stop
itself. Release then reports `cleanup: pending` with `stop_requested:
true`, and the SDK waits up to `RELEASE_TIMEOUT_S` for the GUI to confirm.
If release times out, check the GUI banner and press its Stop before
starting anything else.

In [ ]:
# Caller UUID for start; the GUI uses it as the job ID, so a lost reply can be
# reconciled from this request's own receipt without resending.
START_REQUEST_ID = str(uuid.uuid4())

def add_note(error, message):
    if hasattr(error, "add_note"):  # Python 3.11+
        error.add_note(message)

def lookup_start():
    """Read-only: this request's receipt, or None if the GUI has no such job."""
    try:
        return client.result(START_REQUEST_ID)
    except ControlError as lookup:
        if lookup.application_code == "unknown_run":
            return None
        raise

def reconcile_start(lost):
    """The start reply was lost. Poll its receipt for a bounded time; never resend."""
    deadline = time.monotonic() + RECONCILE_TIMEOUT_S
    while True:
        try:
            found = lookup_start()
        except ControlError as lookup:
            add_note(lost, f"Start receipt could not be read: {lookup}")
            raise lost
        if found is not None or time.monotonic() >= deadline:
            return found
        time.sleep(POLL_INTERVAL_S)

def attempt(step, action):
    """Run one cleanup step; record its error instead of hiding the original."""
    try:
        return action()
    except BaseException as failure:
        cleanup_errors.append((step, failure))
        return None

plan = receipt = finished = stopped = released = None
job_id = None
start_unknown = False  # True while a sent start has no confirmed outcome
primary = None
cleanup_errors = []
# Opt in, for this lease only, to the GUI stopping a running job on release or
# lease expiry. The GUI defaults (both false) are not changed.
try:
    client.acquire(label="Simulation notebook",
                   stop_on_disconnect=True, stop_on_error=True)
    # Read-only check. The notebook never changes the Demo/hardware source.
    source = client.config().get("source")
    if source != "demo" or HARDWARE:
        raise RuntimeError(
            f"Select Demo in the GUI first (source is {source!r}); nothing was started"
        )
    print("GUI source: demo (no hardware)")

    plan = client.invoke("plan", workflow=WORKFLOW)["plan"]
    plan["n_points"] = 5
    plan["interval_s"] = 0.1
    print(json.dumps(plan, indent=2))

    start_unknown = True
    try:
        receipt = client.invoke(
            "start", request_id=START_REQUEST_ID,
            workflow=WORKFLOW, plan=plan, output=str(OUTPUT_CSV),
        )
    except ControlError as lost:
        if not lost.outcome_unknown:
            start_unknown = False  # rejected: no job was created
            raise
        receipt = reconcile_start(lost)
        if receipt is None:
            start_unknown = False  # still unknown, but already looked up
            add_note(lost, (
                f"No job was found for start request {START_REQUEST_ID} within "
                f"{RECONCILE_TIMEOUT_S} s; its outcome is unknown and it was not resent. "
                "If it starts before release, stop_on_disconnect asks the GUI to stop it."))
            raise
        print(f"Start reply was lost; reconciled from the receipt of {START_REQUEST_ID}")
    start_unknown = False
    if receipt.get("job_id") != START_REQUEST_ID:
        raise ControlError(
            "Start receipt does not match the request ID",
            code="validation", request_id=START_REQUEST_ID,
        )
    job_id = START_REQUEST_ID
    print(f"Job receipt: {job_id} · {receipt['status']} · cleanup {receipt['cleanup']}")
    finished = client.wait(job_id, timeout=JOB_TIMEOUT_S, poll_interval=POLL_INTERVAL_S)
except BaseException as error:
    primary = error
    raise
finally:
    if job_id is None and start_unknown:
        # Interrupted before the start outcome was known: look it up read-only.
        if attempt("start receipt lookup", lookup_start) is not None:
            job_id = START_REQUEST_ID
    if job_id is not None and finished is None:
        current = attempt("job receipt", lambda: client.result(job_id))
        if current is None or current.get("terminal") is not True:
            # Stop exactly this job while the lease is still held.
            print(f"Stopping {job_id}")
            stopped = attempt("stop", lambda: client.stop_and_wait(
                job_id, timeout=STOP_TIMEOUT_S, poll_interval=POLL_INTERVAL_S))
            if stopped is not None:
                print(f"Stopped: {stopped['status']} · cleanup {stopped['cleanup']}")
    # With stop_on_disconnect, release asks the GUI to stop any job still running.
    released = attempt("release", lambda: client.release(timeout=RELEASE_TIMEOUT_S))
    if released is not None:
        print(f"Released: cleanup {released['cleanup']} · continuing {released.get('continuing', False)}")
    for step, failure in cleanup_errors:
        message = f"Cleanup error during {step}: {type(failure).__name__}: {failure}"
        print(message, file=sys.stderr)
        if primary is not None:
            add_note(primary, message)
    if cleanup_errors and primary is None:
        step, failure = cleanup_errors[0]
        raise RuntimeError(
            f"Cleanup failed during {step}; check the GUI banner (Stop / Take control)"
        ) from failure

In [ ]:
native = finished["native"]
print(f"Terminal receipt: {finished['status']} · cleanup {finished['cleanup']} · rows {native.get('rows')}")
print(f"CSV written by the GUI: {native.get('output')}")